# Project 1 - Advanced EDA & Feature Engineering
### DecodeLabs Data Science Internship | Batch 2026

**Goal:** Take a raw e-commerce orders dataset and clean it up so it's actually usable for machine learning.

What I'll cover in this notebook:
- Basic EDA to understand the data
- Handling missing values (KNN imputation)
- Detecting and removing outliers using IQR
- Engineering 3+ new features from existing columns
- Saving the final cleaned dataset


## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')
matplotlib.use('Agg')
sns.set_theme(style='darkgrid')

print('Libraries loaded successfully')


## Loading the Dataset

The dataset is e-commerce orders data with 1200 rows and 14 columns.


In [ ]:
df = pd.read_csv('dataset.csv')

print('Shape:', df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


## Step 1: Exploratory Data Analysis

Let me start by understanding what the data looks like — checking for missing values, distributions, and any obvious issues.


### 1.1 Missing Value Analysis

In [ ]:
# checking which columns have null values and how many
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing_count, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print(missing_df)


In [ ]:
# visualizing missing values
fig, ax = plt.subplots(figsize=(9, 4))

missing = df.isnull().sum()
missing = missing[missing > 0]
missing_pct = (missing / len(df) * 100).round(2)

bars = ax.bar(missing.index, missing_pct, color='#e74c3c', edgecolor='white', width=0.4)
for bar, val in zip(bars, missing_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}%', ha='center', fontsize=11, fontweight='bold')

ax.set_title('Missing Values by Column (%)', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Missing %')
ax.set_ylim(0, 35)
plt.tight_layout()
plt.savefig('images/missing_values.png', dpi=150)
plt.show()


### 1.2 Distribution of Numeric Columns

In [ ]:
numeric_cols = ['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=25, color='#3498db', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Numeric Column Distributions', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/distributions.png', dpi=150, bbox_inches='tight')
plt.show()


### 1.3 Categorical Column Breakdown

In [ ]:
cat_cols = ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource']

for col in cat_cols:
    print(f'{col}:')
    print(df[col].value_counts())
    print()


## Step 2: Handling Missing Values

Only `CouponCode` has missing values — about **25.75%** of rows.

Since that's above 20%, I can't just drop rows (too much data loss) or use a simple mean/median (it's a categorical column). So I'll use **KNN Imputation** — it looks at similar rows based on other features and fills in the most likely coupon code.

Here's the decision logic I followed:
- < 5% missing → drop rows
- 5–20% missing → median / group imputation
- > 20% missing → KNN Imputation


In [ ]:
# KNN imputation for CouponCode
# since it's categorical, I'll encode it first, run KNN, then decode back

le_coupon   = LabelEncoder()
le_product  = LabelEncoder()
le_payment  = LabelEncoder()
le_referral = LabelEncoder()

# fit encoders only on known values
known_coupons = df['CouponCode'].dropna().unique()
le_coupon.fit(known_coupons)
le_product.fit(df['Product'])
le_payment.fit(df['PaymentMethod'])
le_referral.fit(df['ReferralSource'])

# build a numeric matrix for KNN to work on
knn_matrix = np.column_stack([
    df['CouponCode'].map(lambda x: le_coupon.transform([x])[0] if pd.notna(x) else np.nan),
    le_product.transform(df['Product']),
    le_payment.transform(df['PaymentMethod']),
    le_referral.transform(df['ReferralSource']),
    df['Quantity'].values,
    df['UnitPrice'].values,
])

imputer = KNNImputer(n_neighbors=5)
knn_imputed = imputer.fit_transform(knn_matrix)

# round the imputed index and decode back to category name
coupon_indices = np.clip(np.round(knn_imputed[:, 0]).astype(int), 0, len(known_coupons) - 1)
df['CouponCode'] = le_coupon.inverse_transform(coupon_indices)

# verify no missing values remain
print('Missing values after imputation:', df.isnull().sum().sum())
print('CouponCode distribution:', df['CouponCode'].value_counts().to_dict())


## Step 3: Outlier Detection Using IQR

Outliers can mess up ML models — especially linear ones — by skewing the distribution.

I'm using the **Interquartile Range (IQR)** method:
- Lower Bound = Q1 - 1.5 × IQR
- Upper Bound = Q3 + 1.5 × IQR

Anything outside those bounds is an outlier.


In [ ]:
# boxplots to visually see outliers before treatment
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#e67e22', alpha=0.7),
                    medianprops=dict(color='black', linewidth=2))
    axes[i].set_title(f'{col} (Before)', fontsize=11, fontweight='bold')
    axes[i].set_ylabel(col)

plt.suptitle('Boxplots Before Outlier Treatment', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/boxplots_before.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# calculating outlier bounds for each numeric column
for col in numeric_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f'{col}: lower={lower:.2f}, upper={upper:.2f}, outliers found={n_out}')


## Step 4: Outlier Treatment — Winsorization

Instead of deleting rows with outliers (which loses data), I'll **cap** the values at the IQR boundaries.
This is called Winsorization — it keeps the row but pulls extreme values back to the statistical limit.

Used `np.clip()` for this — it's fully vectorized and doesn't touch any rows that are already within bounds.


In [ ]:
df_clean = df.copy()

for col in numeric_cols:
    Q1  = df_clean[col].quantile(0.25)
    Q3  = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean[col] = np.clip(df_clean[col], lower, upper)

print('Row count before:', len(df))
print('Row count after: ', len(df_clean))   # should be same — no rows dropped


In [ ]:
# boxplots after winsorization
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df_clean[col], patch_artist=True,
                    boxprops=dict(facecolor='#27ae60', alpha=0.7),
                    medianprops=dict(color='black', linewidth=2))
    axes[i].set_title(f'{col} (After)', fontsize=11, fontweight='bold')
    axes[i].set_ylabel(col)

plt.suptitle('Boxplots After Outlier Treatment (Winsorization)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/boxplots_after.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 5: Feature Engineering

Creating new features that might actually be useful for predicting things like purchase intent or revenue.

| Feature | How it's derived | Why it's useful |
|---|---|---|
| `CartUtilizationRate` | Quantity / ItemsInCart | How much of the cart was actually purchased |
| `HasCoupon` | 1 if coupon used, else 0 | Binary flag — coupons likely affect buying behaviour |
| `OrderDayOfWeek` | extracted from Date | Day-of-week purchase patterns |
| `IsWeekend` | 1 if Sat/Sun, else 0 | Weekend vs weekday shopping habits |
| `OrderMonth` | extracted from Date | Seasonal trends |
| `RevenuePerCartItem` | TotalPrice / ItemsInCart | Revenue efficiency per cart slot |


In [ ]:
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# feature 1: what fraction of cart items were actually bought
df_clean['CartUtilizationRate'] = df_clean['Quantity'] / df_clean['ItemsInCart']

# feature 2: did this order use a coupon? (binary)
df_clean['HasCoupon'] = (df_clean['CouponCode'] != 'NO_COUPON').astype(int)

# feature 3: what day of the week was the order placed
df_clean['OrderDayOfWeek'] = df_clean['Date'].dt.dayofweek  # 0=Monday, 6=Sunday

# feature 4: weekend flag
df_clean['IsWeekend'] = (df_clean['OrderDayOfWeek'] >= 5).astype(int)

# feature 5: which month
df_clean['OrderMonth'] = df_clean['Date'].dt.month

# feature 6: average revenue per cart slot
df_clean['RevenuePerCartItem'] = df_clean['TotalPrice'] / df_clean['ItemsInCart']

print('New features added:')
new_cols = ['CartUtilizationRate', 'HasCoupon', 'OrderDayOfWeek',
            'IsWeekend', 'OrderMonth', 'RevenuePerCartItem']
print(df_clean[new_cols].describe().round(3))


In [ ]:
# quick visual overview of the new features
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

axes[0].hist(df_clean['CartUtilizationRate'], bins=20, color='#9b59b6', edgecolor='white')
axes[0].set_title('CartUtilizationRate', fontweight='bold')
axes[0].set_xlabel('Qty / ItemsInCart')

df_clean['CouponCode'].value_counts().plot(kind='bar', ax=axes[1], color='#e74c3c', edgecolor='white')
axes[1].set_title('CouponCode Distribution (after imputation)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)

df_clean['OrderDayOfWeek'].value_counts().sort_index().plot(
    kind='bar', ax=axes[2], color='#f39c12', edgecolor='white')
axes[2].set_title('Orders by Day of Week', fontweight='bold')
axes[2].set_xlabel('Day (0=Mon, 6=Sun)')
axes[2].tick_params(axis='x', rotation=0)

df_clean['IsWeekend'].value_counts().plot(kind='bar', ax=axes[3], color='#1abc9c', edgecolor='white')
axes[3].set_title('Weekend vs Weekday Orders', fontweight='bold')
axes[3].set_xticklabels(['Weekday', 'Weekend'], rotation=0)

df_clean['OrderMonth'].value_counts().sort_index().plot(
    kind='bar', ax=axes[4], color='#3498db', edgecolor='white')
axes[4].set_title('Orders by Month', fontweight='bold')
axes[4].tick_params(axis='x', rotation=0)

axes[5].hist(df_clean['RevenuePerCartItem'], bins=25, color='#e67e22', edgecolor='white')
axes[5].set_title('RevenuePerCartItem', fontweight='bold')
axes[5].set_xlabel('TotalPrice / ItemsInCart')

plt.suptitle('Engineered Features Overview', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/engineered_features.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 6: Correlation Heatmap

Checking how the features relate to each other, especially to `TotalPrice` which would likely be the target variable for any future ML model.


In [ ]:
corr_cols = ['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice',
             'CartUtilizationRate', 'OrderDayOfWeek', 'IsWeekend',
             'OrderMonth', 'RevenuePerCartItem', 'HasCoupon']

corr = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # only show lower triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Correlation Heatmap (Numeric + Engineered Features)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=150)
plt.show()


In [ ]:
# which features are most correlated with TotalPrice?
print('Correlation with TotalPrice:')
print(corr['TotalPrice'].sort_values(ascending=False).round(3))


## Step 7: Saving the Cleaned Dataset

Saving the final cleaned and feature-engineered dataset as a CSV file.


In [ ]:
df_clean.to_csv('cleaned_dataset.csv', index=False)

print('Cleaned dataset saved!')
print('Final shape:', df_clean.shape)
print('Columns:', list(df_clean.columns))


## Summary

Here's what I did in this notebook:

1. **EDA** — explored the dataset, checked distributions, found that only `CouponCode` had missing values (25.75%)
2. **Missing Value Handling** — used KNN Imputation since the missingness was above 20% and the column was categorical
3. **Outlier Detection** — used IQR method to find outliers in all numeric columns
4. **Outlier Treatment** — used Winsorization (`np.clip`) to cap extreme values without dropping any rows
5. **Feature Engineering** — created 6 new features: `CartUtilizationRate`, `HasCoupon`, `OrderDayOfWeek`, `IsWeekend`, `OrderMonth`, `RevenuePerCartItem`
6. **Correlation Analysis** — heatmap showed `UnitPrice` and `Quantity` are the strongest predictors of `TotalPrice`

The cleaned dataset is ready for machine learning.
